# 15 — MLflow Tracing & Observability

In Notebook 13, we evaluated whether our system behaved correctly.

Now we answer a different question:

> **What actually happened inside one agent execution?**

For a multi-step AI application, the final answer alone is not enough.

We want to see:

```text
User question
    ↓
Routing decision
    ↓
Portfolio lookup
    ↓
Governed risk-function call
    ↓
Claude synthesis
    ↓
Final response
```

MLflow Tracing records this execution as a **trace** made of nested **spans**.

For this POC we will trace a real portfolio-risk workflow using our existing governed assets:

- `investment_ai.portfolio.portfolio_semantic_source`
- `investment_ai.agent.check_portfolio_concentration`
- Claude through Unity Gateway

The goal is observability — not rebuilding our full orchestrator.


## 1. Install tracing dependencies

We use MLflow 3 tracing plus the Anthropic SDK.

Because Databricks Free Edition uses serverless compute, we explicitly enable Anthropic autologging rather than relying on it being enabled automatically.


In [0]:
%pip install -q --upgrade "mlflow[databricks]>=3.14.0" anthropic databricks-sdk
dbutils.library.restartPython()


## 2. Configure MLflow and Claude through Unity Gateway

Manual MLflow spans will trace our custom orchestration and governed tools.

`mlflow.anthropic.autolog()` will automatically capture the native Anthropic model call.

This gives us one connected trace containing both our custom Python logic and the LLM invocation.


In [0]:
import mlflow
from anthropic import Anthropic
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType

EXPERIMENT_NAME = "/Shared/investment_ai_agent_tracing"

PORTFOLIO_SOURCE = "investment_ai.portfolio.portfolio_semantic_source"
RISK_FUNCTION = "investment_ai.agent.check_portfolio_concentration"

MODEL_PROVIDER_SERVICE = "investment_ai.gateway.anthropic_claude"
MODEL = "claude-sonnet-5-5"

POC_AS_OF_DATE = "2026-06-30"
CONCENTRATION_THRESHOLD = 0.10

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_NAME)

# Serverless compute: enable explicitly.
mlflow.anthropic.autolog()

workspace = WorkspaceClient()

auth_header = workspace.config.authenticate()["Authorization"]
databricks_token = auth_header.replace("Bearer ", "", 1)

UNITY_GATEWAY_ANTHROPIC_URL = (
    f"{workspace.config.host.rstrip('/')}/ai-gateway/anthropic"
)

anthropic_client = Anthropic(
    auth_token=databricks_token,
    base_url=UNITY_GATEWAY_ANTHROPIC_URL,
    default_headers={
        "Databricks-Model-Provider-Service": MODEL_PROVIDER_SERVICE,
    },
)

print(f"MLflow experiment: {EXPERIMENT_NAME}")
print("Tracing configured.")


## 3. Trace the routing decision

Our production orchestrator is more sophisticated, but the observability concept is the same.

We create a dedicated `AGENT` span for routing so that, when we inspect the trace, we can see **which path the application selected**.


In [0]:
@mlflow.trace(name="route_question", span_type=SpanType.AGENT)
def route_question(question: str) -> str:
    q = question.lower()

    needs_portfolio = any(
        term in q
        for term in ["portfolio", "holding", "position", "exposure", "largest"]
    )

    needs_risk = any(
        term in q
        for term in ["risk", "concentration", "breach", "threshold", "policy"]
    )

    if needs_portfolio and needs_risk:
        return "portfolio_risk"

    if needs_risk:
        return "risk"

    if needs_portfolio:
        return "portfolio"

    return "general"


## 4. Trace the governed portfolio lookup

The next span queries the portfolio semantic source created earlier in the POC.

For this demo we find the **largest position by portfolio weight** on our POC date.

This is a real data lookup, so the trace will capture the tool input, output, and latency.


In [0]:
@mlflow.trace(name="lookup_largest_position", span_type=SpanType.TOOL)
def lookup_largest_position(as_of_date: str) -> dict:
    result = spark.sql(
        f'''
        SELECT
            portfolio_id,
            portfolio_name,
            security_id,
            company_name,
            sector,
            market_value,
            position_weight
        FROM {PORTFOLIO_SOURCE}
        WHERE as_of_date = DATE '{as_of_date}'
        ORDER BY position_weight DESC
        LIMIT 1
        '''
    ).collect()

    if not result:
        raise ValueError(f"No portfolio data found for {as_of_date}")

    row = result[0]

    return {
        "portfolio_id": row["portfolio_id"],
        "portfolio_name": row["portfolio_name"],
        "security_id": row["security_id"],
        "company_name": row["company_name"],
        "sector": row["sector"],
        "market_value": float(row["market_value"]) if row["market_value"] is not None else None,
        "position_weight": float(row["position_weight"]) if row["position_weight"] is not None else None,
    }


## 5. Trace the governed risk-policy check

Now we call the Unity Catalog function from Notebook 11.

This is an important span because the LLM is **not deciding BREACH or OK**.

The governed function remains the authoritative policy calculation, while MLflow records that the tool was invoked and what it returned.


In [0]:
def _sql_escape(value: str) -> str:
    return value.replace("'", "''")


@mlflow.trace(name="check_concentration_policy", span_type=SpanType.TOOL)
def check_concentration_policy(
    portfolio_id: str,
    security_id: str,
    as_of_date: str,
    threshold: float = 0.10,
) -> dict:

    portfolio_id_sql = _sql_escape(portfolio_id)
    security_id_sql = _sql_escape(security_id)

    result = spark.sql(
        f'''
        SELECT
            portfolio_id,
            portfolio_name,
            security_id,
            company_name,
            position_weight,
            concentration_threshold,
            risk_status,
            excess_weight
        FROM {RISK_FUNCTION}(
            '{portfolio_id_sql}',
            DATE '{as_of_date}',
            {threshold}
        )
        WHERE security_id = '{security_id_sql}'
        LIMIT 1
        '''
    ).collect()

    if not result:
        raise ValueError(
            f"No risk result found for portfolio={portfolio_id}, security={security_id}"
        )

    row = result[0]

    return {
        "portfolio_id": row["portfolio_id"],
        "portfolio_name": row["portfolio_name"],
        "security_id": row["security_id"],
        "company_name": row["company_name"],
        "position_weight": float(row["position_weight"]),
        "concentration_threshold": float(row["concentration_threshold"]),
        "risk_status": row["risk_status"],
        "excess_weight": float(row["excess_weight"]),
    }


## 6. Trace the final Claude synthesis

Claude receives the **already-governed facts** and explains them in natural language.

Because Anthropic autologging is enabled, the native `messages.create()` call is automatically captured as an LLM span inside the same trace.


In [0]:
@mlflow.trace(name="synthesize_risk_answer", span_type=SpanType.CHAIN)
def synthesize_risk_answer(
    question: str,
    position: dict,
    risk_result: dict,
) -> str:

    context = f'''
User question:
{question}

Largest position:
Company: {position["company_name"]}
Portfolio: {position["portfolio_name"]}
Portfolio weight: {position["position_weight"]:.2%}

Governed concentration-policy result:
Threshold: {risk_result["concentration_threshold"]:.2%}
Status: {risk_result["risk_status"]}
Excess weight: {risk_result["excess_weight"]:.2%}
'''

    response = anthropic_client.messages.create(
        model=MODEL,
        max_tokens=600,
        system=(
            "You are an Investment AI Portfolio Risk Assistant. "
            "Explain the supplied governed portfolio-risk result clearly and concisely. "
            "Do not recalculate or override the governed BREACH/OK result."
        ),
        messages=[
            {
                "role": "user",
                "content": context,
            }
        ],
    )

    return "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )


## 7. Run one fully traced workflow

The root `AGENT` span connects all of the nested spans.

When this function runs, the trace should look conceptually like:

```text
investment_ai_observability_demo
├── route_question
├── lookup_largest_position
├── check_concentration_policy
└── synthesize_risk_answer
    └── Anthropic model call
```

This is the key value of tracing: we can inspect the **execution path**, not only the final response.


In [0]:
@mlflow.trace(
    name="investment_ai_observability_demo",
    span_type=SpanType.AGENT,
)
def run_observability_demo(question: str) -> dict:

    route = route_question(question)

    if route != "portfolio_risk":
        return {
            "route": route,
            "answer": "This tracing demo is configured for a portfolio-risk question.",
        }

    position = lookup_largest_position(POC_AS_OF_DATE)

    risk_result = check_concentration_policy(
        portfolio_id=position["portfolio_id"],
        security_id=position["security_id"],
        as_of_date=POC_AS_OF_DATE,
        threshold=CONCENTRATION_THRESHOLD,
    )

    answer = synthesize_risk_answer(
        question=question,
        position=position,
        risk_result=risk_result,
    )

    return {
        "route": route,
        "position": position,
        "risk_result": risk_result,
        "answer": answer,
    }


QUESTION = (
    "Does the largest holding in our portfolio breach the 10% "
    "concentration policy? Explain the result."
)

result = run_observability_demo(QUESTION)

print("Route:", result["route"])
print()
print(result["answer"])


## 8. Inspect the trace in Databricks

Now open the trace visually:

```text
Databricks Workspace
    ↓
Experiments
    ↓
investment_ai_agent_tracing
    ↓
Traces
    ↓
Open the newest trace
```

Inspect the **span tree** from top to bottom.

For each span look at:

- input,
- output,
- duration,
- status / exception,
- parent-child relationship.

For the Anthropic span also inspect the captured model invocation metadata.

This is where observability becomes useful for debugging.

If the final answer were wrong, we could ask:

```text
Was routing wrong?
Did the portfolio lookup return the wrong security?
Did the risk function return BREACH or OK?
Did Claude receive the correct governed result?
Which step took the most time?
Did any step fail?
```


## 9. Optional — list recent traces programmatically

The UI is the best place for the video, but MLflow traces can also be queried programmatically.

This becomes useful later for monitoring many executions rather than opening traces one by one.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

recent_traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=20,
)

display(recent_traces)


# ✅ MLflow Tracing & Observability complete

Evaluation and tracing answer two different questions:

```text
MLflow Evaluation
"Was the behavior correct?"

MLflow Tracing
"What happened during execution?"
```

Together they give us the foundation for agent quality:

```text
User request
    ↓
Trace the execution
    ↓
Inspect agents + tools + model calls
    ↓
Evaluate quality
    ↓
Find regressions / failures / latency
    ↓
Improve the system
```

In production, the same pattern will wrap the complete packaged Multi-Agent Orchestrator so one trace can show Research, Portfolio, Risk, tools, model calls, and final synthesis in a single execution tree.

### Next

**Lakebase — Agent State & Persistent Memory**

That is where we will give the application durable state so later turns can understand conversation context without stuffing the entire chat history into every prompt.
